In [0]:
import os

current_path = os.getcwd()
repo_name = "Grupo7-Setor-de-Seguros"

if repo_name in current_path:
    root_path = current_path.split(repo_name)[0] + repo_name
else:
    root_path = os.path.dirname(os.path.dirname(os.getcwd()))

caminho_arquivo = f"{root_path}/data/processed/prata/seguros_sinistros.csv"

print(f"Diretório Raiz: {root_path}")
print(f"Arquivo Alvo: {caminho_arquivo}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, BooleanType, DateType, LongType

schema_seguros = StructType([
    StructField("nome_contratante", StringType(), True),
    StructField("estado_contratante", StringType(), True),
    StructField("data_contratacao", DateType(), True),
    StructField("valor_pagamento", DoubleType(), True),
    StructField("valor_premio", DoubleType(), True),
    StructField("nome_beneficiario", StringType(), True),
    StructField("status_apolice", StringType(), True),
    StructField("Cobertura1", StringType(), True),
    StructField("Valor Cob 1", DoubleType(), True),
    StructField("Cobertura2", StringType(), True),
    StructField("Valor Cob 2", DoubleType(), True),
    StructField("Cobertura3", StringType(), True),
    StructField("Valor Cob 3", DoubleType(), True),
    StructField("capital_segurado", DoubleType(), True),
    StructField("tipo_sinistro", StringType(), True),
    StructField("valor_sinistro", DoubleType(), True),
    StructField("quem_forma_beneficiados", StringType(), True),
    StructField("status_seguro", StringType(), True),
    StructField("regiao_sinistro", StringType(), True),
    StructField("REGIAO", StringType(), True),
    StructField("SEXO", StringType(), True),
    StructField("TRIMESTRE", IntegerType(), True),
    StructField("ACIMA_DE_3_QUARTIL_PREMIO", BooleanType(), True),
    StructField("ABAIXO_DE_1_QUARTIL_PREMIO", BooleanType(), True),
    StructField("ACIMA_DE_3_QUARTIL_CAPITAL", BooleanType(), True),
    StructField("ABAIXO_DE_1_QUARTIL_CAPITAL", BooleanType(), True),
    StructField("QTD_ACIDENTES_POR_NOME_SEGURADO", LongType(), True),
    StructField("QTD_ACIDENTES_POR_NOME_CONTRATANTE", LongType(), True),
    StructField("RAZÃO_PAGAMENTO_PREMIO", DoubleType(), True),
    StructField("RAZÃO_PAGAMENTO_CAPITAL", DoubleType(), True)
])


df_seguros = spark.read \
    .format("csv") \
    .schema(schema_seguros) \
    .option("header", "true") \
    .option("sep", ",") \
    .option("dateFormat", "yyyy-MM-dd") \
    .load(caminho_arquivo)

df_seguros.printSchema()
display(df_seguros)

1. Região vs. Valores (Capital e Sinistro)

- Ticket Médio: 
  - A região X vende apólices mais caras (Capital Segurado maior) do que a região Y?

- Severidade do Risco: 
  - Quando ocorre um acidente na região X, o custo (Valor Sinistro) tende a ser mais alto?

- Previsibilidade (Desvio Padrão):
  - Desvio Padrão Baixo: Os acidentes custam quase sempre a mesma coisa (fácil de prever reserva).
  - Desvio Padrão Alto: Existem acidentes muito baratos e outros catastróficos (risco maior para o caixa).

In [0]:
from pyspark.sql import functions as F

df_analise_valores = df_seguros.groupBy("REGIAO").agg(
    # Capital Segurado
    F.avg("capital_segurado").alias("Media_Capital"),
    F.stddev("capital_segurado").alias("DesvioPadrao_Capital"),
    
    # # Vamos filtrar apenas quem teve sinistro para calcular a média
    F.avg(F.when(F.col("valor_sinistro") > 0, F.col("valor_sinistro"))).alias("Media_Custo_Sinistro"),
    F.stddev(F.when(F.col("valor_sinistro") > 0, F.col("valor_sinistro"))).alias("DesvioPadrao_Sinistro")
).orderBy("REGIAO")

display(df_analise_valores)

Databricks visualization. Run in Databricks to view.

Resultado
- Não há variações significativas entre as regiões.
---

2. Região vs. Status (Distribuição Proporcional)

Retenção:
- Qual região tem a maior taxa de cancelamento ou não renovação?

Inadimplência/Problemas: 
- Se houver status como "Suspenso", onde eles se concentram?

Comparativo: 
- A região X cancela duas vezes mais que a Y?

In [0]:
# Total de contratos por região
df_total_regiao = df_seguros.groupBy("REGIAO").count().withColumnRenamed("count", "total_regiao")

# Pivotar status em colunas
df_pivot_status = df_seguros.groupBy("REGIAO") \
    .pivot("status_apolice") \
    .count() \
    .na.fill(0) # Substitui null por 0 onde não houver aquele status

# Juntar com o total e calcular percentuais
df_status_percentual = df_pivot_status.join(df_total_regiao, "REGIAO")

colunas_status = [c for c in df_pivot_status.columns if c != "REGIAO"]
exprs = [F.col("REGIAO")] + \
        [(F.col(c) / F.col("total_regiao") * 100).alias(f"Porcentagem {c}") for c in colunas_status]

display(df_status_percentual.select(*exprs))

Databricks visualization. Run in Databricks to view.

### Resultado
- todas as regiões tem a mesma distribuição de seguros ativos, cancelados e suspensos

---
### Região vs. Razão Pagamento/Prêmio

- Razão Alta: 
  - Cliente recebeu muito mais do que pagou (Prejuízo para a seguradora).
- Razão Baixa: 
  - Cliente pagou e usou pouco (Lucro para a seguradora).

- Qual é a região mais lucrativa?

In [0]:
# Mediana: percentile_approx(coluna, 0,5)
df_rentabilidade = df_seguros.groupBy("REGIAO").agg(
    F.avg("RAZÃO_PAGAMENTO_PREMIO").alias("Media Razao"),
    F.percentile_approx("RAZÃO_PAGAMENTO_PREMIO", 0.5).alias("Mediana Razao"),
    F.max("RAZÃO_PAGAMENTO_PREMIO").alias("Maximo Razao") # Para ver o pior caso
).orderBy("Mediana Razao")

display(df_rentabilidade)

### Resultado
- Todas as regiões têm uma razão parecida